<!--
File: technical_memory.ipynb
Description: Running technical decision log for The Village. Records architectural decisions, tradeoffs, and gotchas over the project's lifetime.
Author: Jose-Jorge HERNANDEZ
Company: Parlee Conseiller, Inc.
Date: 2026-09-07
Last edit date: 2026-10-19
Version: 4.4.0
-->

# Technical Memory — The Village

This notebook is a running log of *why* — design decisions, tradeoffs weighed and rejected, gotchas hit and how they were resolved. It is not a changelog of *what* changed (`CHANGELOG.md`) and not a bug log (`docs/BUGS.md`, once one exists) — see each project's own docs for those.

## 2026-09-07 — Documentation overhaul & project history recovery

### Context
The repo's docs (`README.md`, `TODO.md`, `ARCHITECTURE.md`, `CHANGELOG.md`, `GDD.md`) had drifted out of sync with both each other and the actual codebase, and with a much longer project history that lived entirely outside the git repo (in school coursework going back to Feb 2024).

### Decision 1 — Resolving the TODO.md vs CLAUDE.md phase-completion conflict
`CLAUDE.md` claimed "all 6 phases done" (movement/combat, rooms, items, main menu, pause menu, real sprites) while `TODO.md` still showed most of those as unchecked, using an older, differently-numbered phase scheme (0–8) that was never updated after those milestones actually shipped.

**Resolution:** verified directly against the file tree (`scenes/ui/MainMenu.tscn`, `GameOver.tscn`, `HealthUI.tscn`, `scenes/world/rooms/Room1_Tutorial.tscn` + `Room2_Challenge.tscn`, `scenes/items/HeartPickup.tscn`, wired sprites `tileset_forest.png`/`slime.png`/`sword.png`) — all exist. `CLAUDE.md` was correct; `TODO.md` was stale. Rewrote `TODO.md` onto `CLAUDE.md`'s phase numbering so the two documents can't drift apart the same way again.

### Decision 2 — Reconstructing an accurate CHANGELOG from git history instead of the vague `[Unreleased]` block
The changelog only had one dated entry (`[0.0.1]` — 2026-05-25) with everything else bucketed under `[Unreleased]`. Real commit history was available and more precise, so it was used to rebuild dated release entries.

### Decision 3 — Extended lore preserved as a separate `LORE.md`, not merged into `GDD.md`
School research (see below) surfaced a much richer original narrative (protagonist Joe, antagonist Caver, four dimensions) that the current `GDD.md` explicitly scopes *out* of the prototype. Rather than re-inflate the lean prototype GDD, the fuller narrative was captured in a new root-level `LORE.md`, marked as dormant/future-vision material — preserves it for later without disturbing the working, intentionally-small current scope.

### Decision 4 — Asset migration from the school folder, with a licensing caveat
Copied into `assets/branding/`, `assets/reference/joe/`, and `assets/audio/`:
- Branding art (`thevillage*.png/.ico`) and Joe's character reference art/pixel frames — both the student's own coursework, low risk.
- Audio clips (`bg_music`, `button_click`, `forest`, `person-running-loop-245173`, `retro-jump-3-236683`, `pianochocolate_forest`) made for the earlier Unreal Engine 5 prototype.

**Gotcha to resolve before shipping:** the audio files' actual licenses were not verified during this pass — `person-running-loop-245173` and `retro-jump-3-236683` look like they're from a stock SFX site (ID-suffixed filenames, e.g. Pixabay/Freesound convention) and `pianochocolate_forest.mp3` is named after a real artist ("Pianochocolate") and may not be freely redistributable. None of these should be included in a public/shipped build until their license terms are confirmed.

**Deliberately NOT copied:** `800609_warxwell_16x16-pixel-chars.png` (a third-party pixel-character reference sheet found alongside Joe's art) — this reads as someone else's published asset pack used only as visual reference, not the student's own work, so it was left in the school folder rather than copied into the repo.

In [ ]:
%%bash
# Command used to reconstruct the dated CHANGELOG entries from real commit history
# (run from the repo root)
git log --all --date=short --stat --pretty=format:"=== %ad | %s ===" --reverse

### Project history summary (for future sessions)

The Village did **not** start as this Godot repo. Full lineage, recovered from `H:\OneDrive\Office\School\UVM`:

| When | Course | What happened |
|---|---|---|
| 2024-02 | Estructuras Narrativas para Videojuegos | Concept originates — first GDD draft, gothic medieval village, isometric action-RPG |
| 2024-06 | Diseño de Personajes | Protagonist Joe fully designed — character sheet + 8-frame pixel sprite |
| 2024-08 | Diseño de Escena y Componentes | Worldbuilding expands — level map, moodboards, HUD/item art |
| 2024-11 to 2024-12 | Motor de Videojuegos | Built as a working Unreal Engine 5 prototype across 9 graded assignments (movement, collision, rigging, level blockout, menu, sound, AI, save system) |
| *(~17-month gap, no documented activity)* | | |
| 2026-05-25 → present | — | Rebuilt from scratch in Godot 4.6, deliberately scoped down to a lean top-down prototype (no story, single enemy type, no saves) — this repo |

The current repo is a clean-slate reboot in a new engine, not a continuation of the UE5 codebase — no code or assets carried over directly (the UE5 project files were never located in the school folder, only its documentation, videos, and a handful of standalone art/audio assets).

## 2026-10-15 — Rebuild on the Chapel of Dreams architecture (Phases 0 and 1)

### Context
The Godot prototype (two rooms, slimes, placeholder Joe) had been dormant since September. Meanwhile game 2 of the trilogy, *Chapel of Dreams*, was built to beta with a much stronger method: generated art from one palette, manifests read at runtime, `PlayerTuning`-style constant files, and headless scripted test suites. The owner asked to build this game for real, "same mechanics, visuals" as *A Link to the Past*.

### Decisions (owner-confirmed where noted)
1. **Rebuild in place, not extend** (owner). The prototype's autoload web (EventBus/GameManager/SceneManager/InputHandler) did not fit Chapel's patterns, and its art was placeholder. Rebuilding on Chapel's base reuses ~1,000 lines of proven code (HitBox/HurtBox/Health/StateMachine/SpriteStrips/HitFlash) and its whole tooling. The prototype stays in git history (`cec47ca`).
2. **384×216, not ALttP's 256×224** (owner). Consistency across the trilogy won over authenticity. Consequence: a screen is 24×13.5 tiles, so Phase 3's screen-scrolling areas will be 24×14 tiles with half a row hidden behind the HUD band, or the HUD overlays the field — to decide in Phase 3.
3. **Light/Dark mirror worlds** (owner). The four lore dimensions become Dark World regions; `palette.py` already carries a `DW_*` set so the Dark World can be the same generators with a palette swap.
4. **Feet-origin actors with a feet-only collision box** (10×6). This is how ALttP lets Link's head overlap the tile above a wall, and it makes y-sorting trivial (sort by origin).
5. **48×48 frames for Joe.** The sword sweeps around him; drawing it into the frame (instead of a separate rotated sprite) keeps the art exact and lets the manifest carry the hit window.
6. **Runtime TileSet from a JSON manifest** instead of a hand-maintained `.tres`: the generator is the single source of truth for which tiles are solid.
7. **Corner nudge via `test_move`**: when a single-axis step is fully blocked, probe 1..6 px sideways; the first clear offset slides Joe that way at walk speed. Diagonal input never nudges (the slide already handles it).

### Gotchas met
- Case-insensitive filesystem: `Tools/` vs `tools/` produced a *file* named `tools` (see `docs/BUGS.md`).
- `Input.action_press` is seen on the next physics frame: tests wait 2 frames.
- Naming a Node method `_set` would override `Object._set`.


In [ ]:
# The corner-nudge core (src/player/Player.gd, v1.0.0), kept here because its tuning is a feel decision.
func _try_corner_nudge(step: Vector2, delta: float) -> void:
	var perp := Vector2(step.y, step.x).abs()
	for k in range(1, PlayerTuning.CORNER_NUDGE_PX + 1):
		for side in [-1.0, 1.0]:
			var off: Vector2 = perp * side * float(k)
			if not test_move(global_transform.translated(off), step):
				move_and_collide(perp * side * PlayerTuning.NUDGE_SPEED * delta)
				return


## 2026-10-15 — Phase 2: combat core

### Decisions
1. **Shield as a passive cone, not a button.** ALttP's shield blocks automatically while Link faces the projectile. Joe's `_on_hurt` asks `HitBox.blockable` and `blocks_from(source)` (dot product with the facing >= 0.6, i.e. a ~106 degree cone) and only in `idle`/`walk`/`charge`. Swinging, spinning, lifting or carrying leaves him open — the same trade-off ALttP makes.
2. **Projectiles tell the target, the target tells the projectile.** `HurtBox` emits `hurt` *before* `HitBox.landed`, so Joe can call `deflect()` on the rock first and the rock's own `landed` handler then sees `deflected` and stays alive to bounce away. No extra signal needed.
3. **Hit-stop is per actor (`freeze`), not `Engine.time_scale`.** Kept from Chapel: it stays deterministic in tests and does not stall the camera shake, HUD or other actors. Joe pauses his `StateMachine` node's physics process during a freeze.
4. **Thrown objects move on the ground, only the sprite flies.** The body and HitBox travel at ground level (walls and enemies are hit where they stand) while the sprite follows a falling parabola from carry height — ALttP's model, and it keeps collision honest.
5. **Spin as one shared strip with per-facing start frames.** Frame *i* holds the sword at i×45°; the manifest maps each facing to its first frame. One 8-frame strip instead of four, and `flip_h` is turned off for the spin because the left-facing frames are already drawn mirrored.
6. **Enemies overlap Joe and each other.** Like ALttP, enemy bodies collide only with the world and props; damage is by touch. It avoids enemies body-blocking Joe in corridors.
7. **A minimal HUD now (hearts + gems) instead of in Phase 4.** Without it, damage and drops were invisible to a playtester; the full HUD remains Phase 4.

### Gotchas
- In `-s` test scripts the autoloads are reached with `root.get_node(...)`; a static reference compiles before they exist (Chapel lesson, applied up front, no failure hit).
- A windowed capture with live guards produced shots of Joe blinking under a guard: captures now switch his hurt box off.


In [ ]:
# Shield check (src/player/Player.gd v2.0.0)
func blocks_from(source_position: Vector2) -> bool:
	if not fsm.current_state in PlayerTuning.SHIELD_STATES:
		return false
	var to_src := source_position - (global_position + PlayerTuning.BODY_CENTER)
	if to_src.length() < 0.01:
		return false
	return to_src.normalized().dot(Vector2(facing)) >= PlayerTuning.SHIELD_DOT


## 2026-10-15 — Phase 3: overworld engine

### Decisions
1. **One global coordinate system for the overworld.** Each area JSON carries `origin_tiles`; neighbours share edges. Crossing an edge needs no coordinate translation: Joe keeps walking, the World notices his feet left the current rect and asks `overworld_at(feet)` which area owns that point. This also made the lint simple (a global tile -> area map).
2. **Only one area alive at a time** (two during the 0.6 s slide). Cheap, and it gives ALttP's behaviour for free: enemies and bushes reset when you come back.
3. **The camera belongs to the World, not to Joe.** It is clamped per area, centred in rooms smaller than the view, and tweened during scrolls. Test arenas keep the simple child camera.
4. **Ledges are solid tiles plus a flag.** Rather than one-way collision (which Godot defines along a body's local up, awkward top-down), a ledge cell is solid and carries a `ledge` custom-data flag; walking south into it switches Joe to a `hop` state that flies him over the ledge cells without collision. The lint's reachability search understands the one-way edge.
5. **Doors are walkable cells in the house's footprint**, triggered by Joe's feet entering the cell while facing up; exit mats trigger when facing down. No Area2D triggers to keep in sync with the tiles.
6. **Generator owns pairing.** `gen_areas.py` links each house door with its interior's mat in both directions, so a door can never lead to a room without a way back (and the lint re-checks it).
7. **Screen = 24x14 tiles.** 216 px is 13.5 tiles; a 14-tile screen lets the camera scroll 8 px inside one screen rather than leaving half-tiles at area edges.

### Gotchas
- A new `class_name` made the headless suite hang (not fail) until `--import` ran; all Godot runs are now wrapped in `timeout`.
- Nine-slice autotiling must treat off-area cells as the same terrain, or every road shows a grass lip at area seams.
- My first capture script released the stick one frame before Joe reached the edge - a reminder that a windowed shot that "shows nothing happened" needs the timing checked before the code is blamed.


In [ ]:
# How the World finds where Joe went when he leaves an area (src/world/World.gd v1.0.0)
func overworld_at(p: Vector2) -> String:
	for aid in areas:
		if areas[aid]["kind"] == "overworld" and rect_of(aid).has_point(p):
			return aid
	return ""


## 2026-10-15 — Phase 4: HUD, menus, saves, dialogue

**Pausing model.** Every modal screen (dialogue, pause menu, item menu, Game Over) sets `get_tree().paused` and
lives on a CanvasLayer with `PROCESS_MODE_ALWAYS`. The alternative — a "busy" flag every actor checks — was rejected:
enemies, pickups' blink timers, Juice hit-stops and the HUD caption tween would each need to know about it. With a
real pause the world simply stops; only `Audio` (already `ALWAYS`) and `MenuList` (sets itself `ALWAYS`) needed care.

**Talk before lift, and the 0.15 s cooldown.** Interact both talks and lifts (ALttP's A button). Talking wins
because a pot next to a villager must not block the conversation. The box closes on the *same* button that opened
it; the box consumes the event in `_unhandled_input` and unpauses, but Joe's state machine polls
`Input.is_action_just_pressed("interact")` in the same frame and would immediately re-open the talk. Freezing Joe
for `TALK_COOLDOWN` after `closed` makes the polled press expire first (a test checks it).

**Talk points instead of physics queries.** Villagers and signs answer a tiny contract (`talk_point`, `talk`, `rest`)
and Joe measures distance from a probe one tile ahead — the same pattern as `liftable`. Boni behind her counter
just moves her talk point one tile forward (`across`), so the counter needs no special "talk-through" tile.

**Saving at the area's entry point.** The save stores where Joe *entered* the current area, not where he stands.
A loaded game therefore never starts inside a wall, mid-ledge-hop or next to an enemy, and Game Over's CONTINUE
uses the same point. Facing at entry is stored too so Joe looks into the room he walked into.

**Dialogue as data with first-meeting flags.** `dialogue.json` entries have `pages`, and optionally `once` + `flag`:
the first talk shows `once` and sets the flag in `GameState.flags`, which is saved — so story progress and "have I
met Tato" are the same mechanism. The area generator's lint loads the same file, so a typo in a sign key fails the
build instead of showing an empty box.

**Per-effect SFX level targets.** The old self-check accepted only peaks between -6.1 and -2.9 dB, which would have
forced the menu tick and text blip to be as loud as a sword hit. Each effect now has its own target in the `SFX`
table and the check is `|peak - target| <= 0.6 dB`.

**Gotchas paid for this phase** (also in `docs/BUGS.md`): `round()` returns a Variant (inference warning = error
here; use `roundf`); a `-s` test that names an autoload-using class statically fails to compile and silently
half-builds the World (read constants through the instance); long inline heredocs break the shell tool, so patches
were written as script files.

The cell below cross-checks the generated areas against the dialogue file and the villager sprite manifest — the
same rule the `gen_areas.py` lint enforces — and prints which text each villager and sign uses.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
dialogue = json.load(open(os.path.join(ROOT, 'assets', 'data', 'dialogue.json'), encoding='utf-8'))
sprites = set(json.load(open(os.path.join(ROOT, 'assets', 'sprites', 'npcs', 'npcs.json'), encoding='utf-8'))['npcs'])
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))

problems = []
for aid in index['areas']:
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    for n in a.get('npcs', []):
        ok = n['id'] in dialogue and n['id'] in sprites
        print(f"{aid:12s} villager {n['id']:15s} at {n['cell']} -> {dialogue.get(n['id'], {}).get('name', '?')}")
        if not ok:
            problems.append((aid, n['id']))
    for s in a.get('signs', []):
        print(f"{aid:12s} sign     {s['text']:15s} at {s['cell']}")
        if s['text'] not in dialogue:
            problems.append((aid, s['text']))
print('problems:', problems or 'none')

## 2026-10-15 — Phase 5: Items I

**One item-get path.** Chests, villager gifts, loose heart pieces and shop purchases all end in
`World.item_get(kind)`: `grant()` changes the state, Joe `hold_up()`s the icon, the fanfare plays and the kind's
text is shown; the dialogue's `_after_dialogue` callable lowers the item. Having one path is what made Janga's gift
cheap: `DialogueData.conversation()` returns `{pages, gift}` and the World chains `item_get` after the lines.

**Interactions dispatched by class, not new contracts.** Chests, key blocks and wares joined the existing
`talkable` group (`talk_point`/`talk`/`rest`) so Joe's probe finds them for free; the World branches on
`is Chest` / `is KeyBlock` / `is ShopItem` before plain talk. A separate "interactable" contract was considered and
rejected: it would have duplicated the probe and the priority rules (talk before lift) for no gain.

**Stun as a HitBox element.** The boomerang reuses the combat contract with `damage = 0, element = "stun"`;
`BaseEnemy._on_hurt` checks the element first. Enemies need no new detection code, and later elements (fire rod,
ice rod) slot into the same switch.

**Bombs hurt Joe through the normal pipeline.** A bomb carries two HitBoxes - one on PlayerHitBox (enemies,
bushes) and one on EnemyHitBox (Joe, not blockable) - instead of special-casing Joe. Cracked rocks are a group
(`bombable`) with `blast()`, so later bombable walls and the hammer can reuse it.

**World persistence as flags.** Every one-time thing in the world is a GameState flag named
`<kind>:<area>:<x>,<y>` (wares: `ware:<id>`), checked when an area is built. Flags were already saved, so chests,
rocks, locks and heart pieces survive area changes and saves with no new save fields. Torches deliberately do NOT
persist (ALttP rooms reset); a chest they revealed is remembered once opened.

**Lint knows about opening things.** Reachability treats key blocks and cracked rocks as open, and the lint
separately requires keys >= key blocks and torches for every hidden chest, so a sealed area cannot ship.

**Gotchas:** `hidden` is a CanvasItem signal (renamed the chest flag `concealed`); `filter()` returns an untyped
Array (type it, or the headless suite hangs on the parse error); `HurtBox.recheck()` must not query overlaps while
monitoring is off; tests must re-face Joe after an item-get (he faces down) and wait out the 0.2 s use pose.

The cell below lists every one-time object in the generated world with the flag that remembers it - the same names
the World and the saves use.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
keys = locks = 0
for aid in index['areas']:
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    for o in a.get('objects', []):
        x, y = o['cell']
        extra = ' -> ' + o['item'] + (' (hidden)' if o.get('hidden') else '') if o['kind'] == 'chest' else ''
        print(f"{o['kind']}:{aid}:{x},{y}{extra}")
        keys += o.get('item') == 'small_key'
        locks += o['kind'] == 'key_block'
    for w in a.get('wares', []):
        print(f"ware:{w['ware']}  ({aid} counter at {w['cell']})")
print('small keys:', keys, ' key blocks:', locks)

## 2026-10-16 — Phase 6: dungeon framework + the Moss Shrine

**Dungeons reuse the overworld engine.** A dungeon is just more areas (`kind: "dungeon"`) on their own room grid,
grouped by a `dungeon` id. `group_of()` / `area_at()` replaced the overworld-only lookup, so ALttP room-to-room
scrolling came for free. The only dungeon-specific scroll tweak is the push: walls are two tiles thick, so Joe
steps 40 px in (14 px would leave him standing inside the doorway - and inside a shutter that closes behind him).

**Doors live in both rooms.** Only the current room exists in the tree, so a door drawn only on one side would let
Joe scroll into the other room straight through a locked door. Every shared door is listed in both rooms with one
id; its state comes from flags (`door:<id>`, or the shutter's `when` flag), so both copies always agree. The boss
door is the exception: it is open when the room is built and shuts once Joe is inside (after the scroll), which
needs no mirror.

**Room state as flags, not live checks only.** A cleared room sets `cleared:<room>` once; its shutter then stays open
even though enemies respawn when the room is rebuilt (otherwise coming back from the room behind the shutter
would trap Joe). Switches are permanent for the same reason.

**Pits on their own physics layer.** Joe must walk into pits (he falls), enemies must not. A second TileSet physics
layer (layer 9) puts a wall polygon on pit cells that only enemies and push blocks collide with; Joe checks the
`pit` custom-data flag under his feet instead.

**Keys per dungeon.** ALttP keys belong to their dungeon. `GameState.keys` became a property over `key_pools`
keyed by the dungeon id ("" = overworld), switched by the World on every area change, so every existing caller
(`add_keys`, key blocks, the HUD) works unchanged.

**Lint that understands progression.** Reachability treats doors, blocks, cracked rocks and heavy rocks as openable,
and separate rules check what opens them: small keys per dungeon >= distinct locked doors, a big key wherever there
is a big door or chest, a switch for every switch-revealed chest, a glove somewhere for heavy rocks.

**Gotchas:** a variable named like a method in the same class breaks the whole class (`reveal`); an Area2D does not
reliably report a StaticBody2D moved by a tween (the block now presses the switch it stops on directly); a reward
spawned under Joe is collected instantly (the seal appears higher up); a rebuilt room frees its nodes, so tests must
fetch doors/switches again.

The cell below walks the shrine's progression from the generated data: which room needs which key or flag.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
print('dungeons:', index.get('dungeons'))
for aid in index['areas']:
    a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
    if a['kind'] != 'dungeon':
        continue
    doors = ['%s %s(%s)' % (o['dir'], o['type'], o.get('when') or o['id']) for o in a['objects'] if o['kind'] == 'door']
    chests = ['%s%s' % (o['item'], ' [' + o['reveal'] + ']' if o.get('reveal') else '') for o in a['objects']
              if o['kind'] in ('chest', 'big_chest')]
    print('%-13s grid %s  doors: %s  chests: %s  %s' % (aid, a['grid'], ', '.join(doors) or '-', ', '.join(chests) or '-',
          a.get('clear_flag', '') or a.get('boss_flag', '')))

## 2026-10-19 — Phase 7: the rest of Act 1

**Dungeons as data plus one helper.** The Moss Shrine was hand-coded as `d1_rooms()`. Rather than copy its inner
closure three times, `make_room(rooms, id, gx, gy, doors, fill, extra)` became the shared builder and `DUNGEONS`
lists every dungeon with its table entry; each gets its own room grid `DUNGEON_STRIDE` tiles apart. Ram's Hollow,
the Frost Vault and the keep are ~25 lines each.

**Boss weaknesses through one hook.** ALttP bosses each need a trick or a weapon. `BaseEnemy._resists(hit)` runs
before stun and damage and makes a resisted hit clank; the three bosses override it with one line each (dazed,
`element == "fire"`, `element == "orb"`). Elements already existed on HitBox (the boomerang's "stun"), so fireballs
and Caver's batted orbs only had to set theirs. No new detection code anywhere.

**The batted orb is a two-box object.** It carries a HitBox (EnemyHitBox layer, hurts Joe) and a HurtBox that
listens to Joe's attacks; when struck it moves its HitBox to the PlayerHitBox layer with element "orb" and homes on
its caster. The shade resists everything else, so the only way to win is the Agahnim rally - with zero special
cases in Joe's code.

**The act ending is a dungeon property.** The keep's table entry has `ending: "act1"` and `seal: -1`; the generic
boss-defeated path checks it and calls `play_ending` (last words -> `EndCard` -> save outside the keep -> title)
instead of spawning a reward. Phase 8 can pick up the Act 2 opening from that save and the `act1:done` flag.

**Generic first, tweak by content.** The seal gate is a "glyphless" object: it sits on plain floor cells (so the
doorway lint and door warps work unchanged) and covers them with a 2x2 solid body until the flag opens it.

**Gotchas:** test scripts must not name autoload-dependent classes statically - that now includes the bosses and
even `AreaBuilder` (its dependency Bush uses Audio); a 32-px talkable needs its talk point where Joe's 1-tile probe
reaches from either half; a room rebuild relights nothing (torches go out), which tests must respect.

The cell below lists every dungeon with its rooms, treasure and boss, straight from the generated data.

In [ ]:
import json, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'technical_memory' else os.getcwd()
areas_dir = os.path.join(ROOT, 'src', 'world', 'areas')
index = json.load(open(os.path.join(areas_dir, 'index.json'), encoding='utf-8'))
BOSSES = {'Y': 'Grand Slime', 'U': 'Iron Ram', 'F': 'Frost Wraith', 'V': "Caver's shade"}
for did, info in index['dungeons'].items():
    rooms, treasure, boss = [], [], '-'
    for aid in index['areas']:
        a = json.load(open(os.path.join(areas_dir, aid + '.json'), encoding='utf-8'))
        if a.get('dungeon') != did:
            continue
        rooms.append(aid)
        treasure += [o['item'] for o in a['objects'] if o['kind'] == 'big_chest']
        for g, name in BOSSES.items():
            if any(g in r for r in a['rows']):
                boss = name
    print('%-5s %-14s seal %2s  rooms %d  big chest %-10s boss %-14s %s' % (
        did, info['name'], info['seal'], len(rooms), ','.join(treasure) or '-', boss, info.get('ending', '')))